# TFT Experiments — Walmart Store Sales Forecasting

ეს notebook არის **leakage-safe, exogenous-feature TFT experiment notebook**.

მთავარი გაუმჯობესებები:
- feature-ები დაყოფილია calendar, markdown, external და static ჯგუფებად;
- prediction-ისთვის იქმნება სრული `unique_id × horizon` future grid;
- validation-only series-ებისთვის გამოიყენება median fallback;
- ჩართულია internal validation, early stopping და start padding;
- feature ablation და tuning კეთდება ერთი `EXPERIMENT` ცვლადის შეცვლით;
- თითოეული run სრულად და მკაფიოდ ილოგება MLflow-ში.


## 1. Setup repo, paths, and packages

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.git"
REPO_DIR = Path("/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", Path.cwd())

Mounted at /content/drive
Working directory: /content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting


In [3]:
!pip install -q mlflow neuralforecast utilsforecast dagshub pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68

## 2. Download Kaggle data if needed

In [4]:
DATA_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    DATA_DIR / "train.csv.zip",
    DATA_DIR / "test.csv.zip",
    DATA_DIR / "features.csv.zip",
    DATA_DIR / "stores.csv",
]

if not all(p.exists() for p in required_files):
    print("Raw Kaggle files not found. Upload kaggle.json when prompted.")
    from google.colab import files
    files.upload()

    Path.home().joinpath(".kaggle").mkdir(exist_ok=True)
    subprocess.run(["cp", "kaggle.json", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)
    subprocess.run(["chmod", "600", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)

    subprocess.run([
        "kaggle", "competitions", "download",
        "-c", "walmart-recruiting-store-sales-forecasting",
        "-p", str(DATA_DIR)
    ], check=True)

    subprocess.run([
        "unzip", "-o",
        str(DATA_DIR / "walmart-recruiting-store-sales-forecasting.zip"),
        "-d", str(DATA_DIR)
    ], check=True)
else:
    print("Raw Kaggle files already exist. Skipping download.")

Raw Kaggle files not found. Upload kaggle.json when prompted.


Saving kaggle.json to kaggle.json


## 3. Run or load common preprocessing

In [5]:
import importlib
import preprocessing
importlib.reload(preprocessing)

from preprocessing import run_pipeline, load_processed, weighted_mae

processed_files = [
    PROCESSED_DIR / "train_part.parquet",
    PROCESSED_DIR / "valid_part.parquet",
    PROCESSED_DIR / "train_full.parquet",
    PROCESSED_DIR / "test_full.parquet",
]

# პირველი გაშვებისას დატოვე True, რათა ძველი cached parquet-ები არ ჩაიტვირთოს.
# მას შემდეგ, რაც ახალი leakage-safe preprocessing წარმატებით შესრულდება,
# შემდეგ run-ებზე შეგიძლია False გახადო.
FORCE_REPROCESS = True

if FORCE_REPROCESS or not all(p.exists() for p in processed_files):
    print("Running common preprocessing from scratch...")
    for p in processed_files:
        if p.exists():
            p.unlink()

    train_part, valid_part, train_full, test_full = run_pipeline(
        data_dir=DATA_DIR,
        out_dir=PROCESSED_DIR,
        months_valid=3,
        save=True,
    )
else:
    print("Loading existing processed parquet files...")
    train_part, valid_part, train_full, test_full = load_processed(PROCESSED_DIR)

print("train_part:", train_part.shape, train_part["Date"].min(), "→", train_part["Date"].max())
print("valid_part:", valid_part.shape, valid_part["Date"].min(), "→", valid_part["Date"].max())
print("train_full:", train_full.shape)
print("test_full:", test_full.shape)
print("Train Store-Dept series:", train_part[["Store", "Dept"]].drop_duplicates().shape[0])
print("Validation Store-Dept series:", valid_part[["Store", "Dept"]].drop_duplicates().shape[0])


Running common preprocessing from scratch...
Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/data/processed
train_part: (428409, 33) 2010-02-05 00:00:00 → 2012-07-20 00:00:00
valid_part: (41463, 32) 2012-07-27 00:00:00 → 2012-10-26 00:00:00
train_full: (476333, 33)
test_full: (115064, 29)
Train Store-Dept series: 3321
Validation Store-Dept series: 3104


## 4. Choose one experiment and configure TFT

თითო run-ზე შეცვალე მხოლოდ `EXPERIMENT`. დანარჩენ პარამეტრებს ეტაპობრივად შემდეგ ვცვლით.


In [6]:
MODEL_NAME = "TFT"

# ============================================================
# CHANGE ONLY THIS VALUE FOR FEATURE ABLATION RUNS
# Options:
#   "calendar_only"
#   "calendar_static"
#   "calendar_static_markdown"
#   "all_exog"
# ============================================================
EXPERIMENT = "all_exog"

# Feature-ablation run-ებზე False დატოვე.
# მხოლოდ საბოლოო გამარჯვებულ configuration-ზე ჩართე True.
RUN_TRAIN_TAIL_CHECK = False

CONFIG = {
    "model": "TFT",
    "model_type": EXPERIMENT,
    "horizon": int(valid_part["Date"].nunique()),
    "input_size": 104,
    "freq": "W-FRI",

    "target_col": "Weekly_Sales",
    "evaluation_target_col": "Weekly_Sales",
    "target_transform": "none",
    "missing_target_strategy": "linear_interpolation",

    "hidden_size": 64,
    "n_head": 4,
    "n_rnn_layers": 2,
    "dropout": 0.05,
    "attn_dropout": 0.0,

    "loss": "MAE",
    "learning_rate": 1e-3,
    "batch_size": 32,
    "valid_batch_size": 64,
    "windows_batch_size": 512,
    "inference_windows_batch_size": 512,
    "max_steps": 4000,
    "num_lr_decays": 3,
    "early_stop_patience_steps": 5,
    "val_check_steps": 250,
    "scaler_type": "robust",
    "start_padding_enabled": True,
    "random_seed": 42,

    "validation_split": "last_3_months",
}

RUN_NAME = (
    f"TFT_{CONFIG['model_type']}"
    f"_input{CONFIG['input_size']}_h{CONFIG['horizon']}"
    f"_hidden{CONFIG['hidden_size']}_heads{CONFIG['n_head']}"
    f"_lr{CONFIG['learning_rate']}_drop{CONFIG['dropout']}"
    f"_scaler{CONFIG['scaler_type']}_steps{CONFIG['max_steps']}"
)

print("RUN_NAME:", RUN_NAME)
CONFIG


RUN_NAME: TFT_all_exog_input104_h14_hidden64_heads4_lr0.001_drop0.05_scalerrobust_steps4000


{'model': 'TFT',
 'model_type': 'all_exog',
 'horizon': 14,
 'input_size': 104,
 'freq': 'W-FRI',
 'target_col': 'Weekly_Sales',
 'evaluation_target_col': 'Weekly_Sales',
 'target_transform': 'none',
 'missing_target_strategy': 'linear_interpolation',
 'hidden_size': 64,
 'n_head': 4,
 'n_rnn_layers': 2,
 'dropout': 0.05,
 'attn_dropout': 0.0,
 'loss': 'MAE',
 'learning_rate': 0.001,
 'batch_size': 32,
 'valid_batch_size': 64,
 'windows_batch_size': 512,
 'inference_windows_batch_size': 512,
 'max_steps': 4000,
 'num_lr_decays': 3,
 'early_stop_patience_steps': 5,
 'val_check_steps': 250,
 'scaler_type': 'robust',
 'start_padding_enabled': True,
 'random_seed': 42,
 'validation_split': 'last_3_months'}

## 5. TFT feature groups and input preparation

TFT feature-ები შეგნებულად იყოფა ჯგუფებად, რათა თითო ჯგუფის რეალური ეფექტი ცალკე გავზომოთ.


In [7]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple, List, Optional
from neuralforecast import NeuralForecast

CALENDAR_FUTR_COLS = [
    "IsHoliday_int",
    "Month",
    "Week_sin",
    "Week_cos",
    "is_superbowl",
    "is_labor_day",
    "is_thanksgiving",
    "is_christmas",
]

MARKDOWN_FUTR_COLS = [
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "MarkDown1_exists", "MarkDown2_exists", "MarkDown3_exists",
    "MarkDown4_exists", "MarkDown5_exists",
]

EXTERNAL_FUTR_COLS = [
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
]

STATIC_COLS = ["Size", "Type_A", "Type_B", "Type_C"]

EXPERIMENT_FEATURES = {
    "calendar_only": {
        "futr": CALENDAR_FUTR_COLS,
        "stat": [],
    },

    "calendar_static": {
        "futr": CALENDAR_FUTR_COLS,
        "stat": STATIC_COLS,
    },

    "calendar_static_markdown": {
        "futr": CALENDAR_FUTR_COLS + MARKDOWN_FUTR_COLS,
        "stat": STATIC_COLS,
    },

    "all_exog": {
        "futr": (
            CALENDAR_FUTR_COLS
            + MARKDOWN_FUTR_COLS
            + EXTERNAL_FUTR_COLS
        ),
        "stat": STATIC_COLS,
    },

    "calendar_markdown": {
        "futr": CALENDAR_FUTR_COLS + MARKDOWN_FUTR_COLS,
        "stat": [],
    },
}

if EXPERIMENT not in EXPERIMENT_FEATURES:
    raise ValueError(f"Unknown EXPERIMENT: {EXPERIMENT}")

FUTR_EXOG_COLS = EXPERIMENT_FEATURES[EXPERIMENT]["futr"]
STAT_EXOG_COLS = EXPERIMENT_FEATURES[EXPERIMENT]["stat"]


def fix_isholiday_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "IsHoliday" not in df.columns:
        if "IsHoliday_x" in df.columns:
            df["IsHoliday"] = df["IsHoliday_x"]
        elif "IsHoliday_y" in df.columns:
            df["IsHoliday"] = df["IsHoliday_y"]
        else:
            raise KeyError("No IsHoliday column found.")
    drop_cols = [c for c in ["IsHoliday_x", "IsHoliday_y"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    return df


def fill_missing_targets_for_series(df: pd.DataFrame, target_col: str, strategy: str) -> pd.DataFrame:
    df = df.sort_values(["Store", "Dept", "Date"]).copy()
    if strategy == "zero":
        df[target_col] = df[target_col].fillna(0)
    elif strategy == "linear_interpolation":
        df[target_col] = (
            df.groupby(["Store", "Dept"])[target_col]
              .transform(lambda s: s.interpolate(method="linear", limit_area="inside"))
        )
        df[target_col] = df[target_col].fillna(0)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")
    return df


def add_tft_features(df: pd.DataFrame) -> pd.DataFrame:
    df = fix_isholiday_columns(df)
    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = pd.to_datetime(df["Date"])
    df["IsHoliday_int"] = df["IsHoliday"].astype(int)

    for t in ["A", "B", "C"]:
        df[f"Type_{t}"] = (df["Type"].astype(str) == t).astype(int)

    required = sorted(set(FUTR_EXOG_COLS + STAT_EXOG_COLS))
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Required TFT features are missing: {missing}")

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df


def make_tft_dfs(
    df: pd.DataFrame,
    config: Dict,
    futr_exog_cols: List[str],
    stat_exog_cols: List[str],
    include_y: bool = True,
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    df = add_tft_features(df)

    if include_y:
        df = fill_missing_targets_for_series(
            df,
            target_col=config["target_col"],
            strategy=config["missing_target_strategy"],
        )
        if config["target_transform"] == "log1p":
            df["y"] = np.log1p(df[config["target_col"]].clip(lower=0))
        elif config["target_transform"] == "none":
            df["y"] = df[config["target_col"]]
        else:
            raise ValueError(f"Unknown target_transform: {config['target_transform']}")
        temporal_cols = ["unique_id", "ds", "y"] + futr_exog_cols
    else:
        temporal_cols = ["unique_id", "ds"] + futr_exog_cols

    nf_df = df[temporal_cols].sort_values(["unique_id", "ds"]).reset_index(drop=True)

    static_df = None
    if stat_exog_cols:
        static_df = (
            df[["unique_id"] + stat_exog_cols]
            .drop_duplicates("unique_id")
            .sort_values("unique_id")
            .reset_index(drop=True)
        )
    return nf_df, static_df


def inverse_target_transform(preds: pd.DataFrame, model_col: str, target_transform: str) -> pd.DataFrame:
    preds = preds.rename(columns={model_col: "pred_transformed"}).copy()
    if target_transform == "log1p":
        preds["pred"] = np.expm1(preds["pred_transformed"])
    elif target_transform == "none":
        preds["pred"] = preds["pred_transformed"]
    else:
        raise ValueError(f"Unknown target_transform: {target_transform}")
    preds["pred"] = preds["pred"].clip(lower=0)
    return preds


def evaluate_wmae(eval_df: pd.DataFrame, config: Dict) -> Dict[str, float]:
    target = config["evaluation_target_col"]
    if eval_df["pred"].isna().any():
        raise ValueError("Predictions still contain NaN values before evaluation.")

    holiday_mask = eval_df["IsHoliday"].astype(bool)
    nonholiday_mask = ~holiday_mask
    metrics = {
        "wmae": float(weighted_mae(
            eval_df[target].fillna(0).values,
            eval_df["pred"].values,
            holiday_mask.values,
        )),
        "holiday_wmae": float(weighted_mae(
            eval_df.loc[holiday_mask, target].fillna(0).values,
            eval_df.loc[holiday_mask, "pred"].values,
            eval_df.loc[holiday_mask, "IsHoliday"].astype(bool).values,
        )) if holiday_mask.any() else np.nan,
        "nonholiday_wmae": float(weighted_mae(
            eval_df.loc[nonholiday_mask, target].fillna(0).values,
            eval_df.loc[nonholiday_mask, "pred"].values,
            eval_df.loc[nonholiday_mask, "IsHoliday"].astype(bool).values,
        )),
    }
    return metrics


def make_complete_future_df(
    nf: NeuralForecast,
    train_nf: pd.DataFrame,
    future_source_df: pd.DataFrame,
    futr_exog_cols: List[str],
    horizon: int,
) -> pd.DataFrame:
    future_grid = nf.make_future_dataframe(df=train_nf)
    future_grid[["Store", "Dept"]] = (
        future_grid["unique_id"].str.split("_", n=1, expand=True).astype(int)
    )

    source = add_tft_features(future_source_df)
    feature_lookup = (
        source[["Store", "ds"] + futr_exog_cols]
        .drop_duplicates(["Store", "ds"])
    )

    futr_df = future_grid.merge(
        feature_lookup,
        on=["Store", "ds"],
        how="left",
        validate="many_to_one",
    )

    missing_counts = futr_df[futr_exog_cols].isna().sum()
    missing_counts = missing_counts[missing_counts > 0]
    if not missing_counts.empty:
        raise ValueError(
            "Missing future covariates after creating complete grid\n"
            + missing_counts.to_string()
        )

    expected_rows = train_nf["unique_id"].nunique() * horizon
    assert len(futr_df) == expected_rows, (len(futr_df), expected_rows)
    assert not futr_df[["unique_id", "ds"]].duplicated().any()

    return (
        futr_df[["unique_id", "ds"] + futr_exog_cols]
        .sort_values(["unique_id", "ds"])
        .reset_index(drop=True)
    )


def add_fallback_predictions(
    eval_df: pd.DataFrame,
    train_df_raw: pd.DataFrame,
    target_col: str,
) -> Tuple[pd.DataFrame, int, int]:
    eval_df = eval_df.copy()
    missing_mask = eval_df["pred"].isna()
    missing_rows = int(missing_mask.sum())
    missing_ids = int(eval_df.loc[missing_mask, "unique_id"].nunique())
    if not missing_rows:
        return eval_df, 0, 0

    clean = train_df_raw.dropna(subset=[target_col])
    store_dept_median = clean.groupby(["Store", "Dept"])[target_col].median()
    dept_median = clean.groupby("Dept")[target_col].median()
    store_median = clean.groupby("Store")[target_col].median()
    global_median = clean[target_col].median()

    fallbacks = []
    for row in eval_df.loc[missing_mask].itertuples():
        value = store_dept_median.get((row.Store, row.Dept), np.nan)
        if pd.isna(value):
            value = dept_median.get(row.Dept, np.nan)
        if pd.isna(value):
            value = store_median.get(row.Store, np.nan)
        if pd.isna(value):
            value = global_median
        fallbacks.append(value)

    eval_df.loc[missing_mask, "pred"] = fallbacks
    return eval_df, missing_rows, missing_ids

print("EXPERIMENT:", EXPERIMENT)
print("Future exogenous:", FUTR_EXOG_COLS)
print("Static exogenous:", STAT_EXOG_COLS)


EXPERIMENT: all_exog
Future exogenous: ['IsHoliday_int', 'Month', 'Week_sin', 'Week_cos', 'is_superbowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'MarkDown1_exists', 'MarkDown2_exists', 'MarkDown3_exists', 'MarkDown4_exists', 'MarkDown5_exists', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
Static exogenous: ['Size', 'Type_A', 'Type_B', 'Type_C']


## 6. Baseline: Store-Dept median

In [8]:
def store_dept_median_baseline(train_df: pd.DataFrame, actual_df: pd.DataFrame, target_col: str):
    train_clean = train_df.dropna(subset=[target_col]).copy()
    med = train_clean.groupby(["Store", "Dept"])[target_col].median()
    global_med = train_clean[target_col].median()

    eval_df = actual_df.copy()
    eval_df["pred"] = [
        med.get((store, dept), global_med)
        for store, dept in zip(eval_df["Store"], eval_df["Dept"])
    ]
    eval_df["pred"] = eval_df["pred"].clip(lower=0)
    return eval_df, evaluate_wmae(eval_df, CONFIG)

baseline_valid_eval, baseline_valid_metrics = store_dept_median_baseline(
    train_part,
    valid_part,
    CONFIG["target_col"],
)

print("Baseline validation WMAE:", baseline_valid_metrics)

Baseline validation WMAE: {'wmae': 2245.1614774879517, 'holiday_wmae': 2518.7794892110587, 'nonholiday_wmae': 2139.7570274307086}


## 7. Train-tail check

In [9]:
# Last CONFIG['horizon'] weeks from train_part, only for a quick overfit/underfit signal.
train_dates = sorted(train_part["Date"].unique())
tail_dates = train_dates[-CONFIG["horizon"]:]

train_inner = train_part[~train_part["Date"].isin(tail_dates)].copy()
train_tail = train_part[train_part["Date"].isin(tail_dates)].copy()

baseline_train_tail_eval, baseline_train_tail_metrics = store_dept_median_baseline(
    train_inner,
    train_tail,
    CONFIG["target_col"],
)

print("train_inner:", train_inner["Date"].min(), "→", train_inner["Date"].max())
print("train_tail:", train_tail["Date"].min(), "→", train_tail["Date"].max())
print("Baseline train-tail WMAE:", baseline_train_tail_metrics)

train_inner: 2010-02-05 00:00:00 → 2012-04-13 00:00:00
train_tail: 2012-04-20 00:00:00 → 2012-07-20 00:00:00
Baseline train-tail WMAE: {'wmae': 2200.873600034413, 'holiday_wmae': nan, 'nonholiday_wmae': 2200.873600034413}


## 8. Build + train TFT

In [10]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE


def build_tft(
    config: Dict,
    futr_exog_cols: List[str],
    stat_exog_cols: List[str],
) -> NeuralForecast:
    model = TFT(
        h=config["horizon"],
        input_size=config["input_size"],
        futr_exog_list=futr_exog_cols or None,
        hist_exog_list=None,
        stat_exog_list=stat_exog_cols or None,
        hidden_size=config["hidden_size"],
        n_head=config["n_head"],
        n_rnn_layers=config["n_rnn_layers"],
        dropout=config["dropout"],
        attn_dropout=config["attn_dropout"],
        loss=MAE(),
        valid_loss=MAE(),
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        batch_size=config["batch_size"],
        valid_batch_size=config["valid_batch_size"],
        windows_batch_size=config["windows_batch_size"],
        inference_windows_batch_size=config["inference_windows_batch_size"],
        num_lr_decays=config["num_lr_decays"],
        early_stop_patience_steps=config["early_stop_patience_steps"],
        val_check_steps=config["val_check_steps"],
        scaler_type=config["scaler_type"],
        start_padding_enabled=config["start_padding_enabled"],
        random_seed=config["random_seed"],
        alias=config["model"],
    )

    print("TFT future exogenous inputs:", futr_exog_cols)
    print("TFT static exogenous inputs:", stat_exog_cols)
    return NeuralForecast(models=[model], freq=config["freq"])


def fit_predict_evaluate(
    train_df_raw: pd.DataFrame,
    actual_df_raw: pd.DataFrame,
    config: Dict,
    label: str,
    futr_exog_cols: List[str],
    stat_exog_cols: List[str],
):
    train_nf, static_df = make_tft_dfs(
        train_df_raw,
        config,
        futr_exog_cols=futr_exog_cols,
        stat_exog_cols=stat_exog_cols,
        include_y=True,
    )

    print("\n", label)
    print("Training columns:", train_nf.columns.tolist())
    print("Training series:", train_nf["unique_id"].nunique())
    print("Training date range:", train_nf["ds"].min(), "→", train_nf["ds"].max())

    nf = build_tft(config, futr_exog_cols, stat_exog_cols)
    fit_kwargs = {
        "df": train_nf,
        "val_size": config["horizon"],
    }
    if static_df is not None:
        fit_kwargs["static_df"] = static_df
    nf.fit(**fit_kwargs)

    if futr_exog_cols:
        futr_df = make_complete_future_df(
            nf=nf,
            train_nf=train_nf,
            future_source_df=actual_df_raw,
            futr_exog_cols=futr_exog_cols,
            horizon=config["horizon"],
        )
        preds = nf.predict(futr_df=futr_df)
    else:
        preds = nf.predict()

    preds = inverse_target_transform(
        preds,
        model_col=config["model"],
        target_transform=config["target_transform"],
    )

    actual = actual_df_raw.copy()
    actual["unique_id"] = actual["Store"].astype(str) + "_" + actual["Dept"].astype(str)
    actual["ds"] = pd.to_datetime(actual["Date"])

    eval_df = actual.merge(
        preds[["unique_id", "ds", "pred", "pred_transformed"]],
        on=["unique_id", "ds"],
        how="left",
    )

    eval_df, missing_prediction_rows, unseen_series = add_fallback_predictions(
        eval_df=eval_df,
        train_df_raw=train_df_raw,
        target_col=config["target_col"],
    )
    eval_df["pred"] = eval_df["pred"].clip(lower=0)
    metrics = evaluate_wmae(eval_df, config)
    metrics["missing_prediction_rows_before_fallback"] = missing_prediction_rows
    metrics["unseen_series_before_fallback"] = unseen_series

    print(f"WMAE: {metrics['wmae']:.4f}")
    print(f"Holiday WMAE: {metrics['holiday_wmae']:.4f}")
    print(f"Non-holiday WMAE: {metrics['nonholiday_wmae']:.4f}")
    print("Fallback rows:", missing_prediction_rows, "| unseen series:", unseen_series)

    return nf, preds, eval_df, metrics


In [11]:
# Sanity check before training
input_check, static_check = make_tft_dfs(
    train_part,
    CONFIG,
    futr_exog_cols=FUTR_EXOG_COLS,
    stat_exog_cols=STAT_EXOG_COLS,
    include_y=True,
)

expected_temporal = ["unique_id", "ds", "y"] + FUTR_EXOG_COLS
assert input_check.columns.tolist() == expected_temporal
if STAT_EXOG_COLS:
    assert static_check is not None
    assert static_check.columns.tolist() == ["unique_id"] + STAT_EXOG_COLS
else:
    assert static_check is None

print("Temporal columns:", input_check.columns.tolist())
print("Static columns:", [] if static_check is None else static_check.columns.tolist())
print("Missing temporal values:", int(input_check.isna().sum().sum()))


Temporal columns: ['unique_id', 'ds', 'y', 'IsHoliday_int', 'Month', 'Week_sin', 'Week_cos', 'is_superbowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'MarkDown1_exists', 'MarkDown2_exists', 'MarkDown3_exists', 'MarkDown4_exists', 'MarkDown5_exists', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
Static columns: ['unique_id', 'Size', 'Type_A', 'Type_B', 'Type_C']
Missing temporal values: 0


## 9. Run validation experiment

In [12]:
# Main validation run — always execute this.
nf_valid, valid_preds, valid_eval, valid_metrics = fit_predict_evaluate(
    train_df_raw=train_part,
    actual_df_raw=valid_part,
    config=CONFIG,
    label=RUN_NAME,
    futr_exog_cols=FUTR_EXOG_COLS,
    stat_exog_cols=STAT_EXOG_COLS,
)

improvement = baseline_valid_metrics["wmae"] - valid_metrics["wmae"]
improvement_pct = 100 * improvement / baseline_valid_metrics["wmae"]

rows = [
    {"split": "baseline_valid", **baseline_valid_metrics},
    {"split": "valid", **valid_metrics},
]

# Train-tail check is intentionally optional because it trains a second TFT.
train_tail_metrics = None
train_tail_eval = None
generalization_gap = np.nan

if RUN_TRAIN_TAIL_CHECK:
    nf_train_tail, train_tail_preds, train_tail_eval, train_tail_metrics = fit_predict_evaluate(
        train_df_raw=train_inner,
        actual_df_raw=train_tail,
        config=CONFIG,
        label=f"{RUN_NAME}_train_tail",
        futr_exog_cols=FUTR_EXOG_COLS,
        stat_exog_cols=STAT_EXOG_COLS,
    )
    generalization_gap = valid_metrics["wmae"] - train_tail_metrics["wmae"]
    rows.insert(0, {"split": "baseline_train_tail", **baseline_train_tail_metrics})
    rows.insert(1, {"split": "train_tail", **train_tail_metrics})

summary = pd.DataFrame(rows)
display(summary)
print("Improvement over baseline:", improvement)
print("Improvement %:", improvement_pct)
if RUN_TRAIN_TAIL_CHECK:
    print("Generalization gap valid - train_tail:", generalization_gap)


INFO:lightning_fabric.utilities.seed:Seed set to 42



 TFT_all_exog_input104_h14_hidden64_heads4_lr0.001_drop0.05_scalerrobust_steps4000
Training columns: ['unique_id', 'ds', 'y', 'IsHoliday_int', 'Month', 'Week_sin', 'Week_cos', 'is_superbowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'MarkDown1_exists', 'MarkDown2_exists', 'MarkDown3_exists', 'MarkDown4_exists', 'MarkDown5_exists', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
Training series: 3321
Training date range: 2010-02-05 00:00:00 → 2012-07-20 00:00:00
TFT future exogenous inputs: ['IsHoliday_int', 'Month', 'Week_sin', 'Week_cos', 'is_superbowl', 'is_labor_day', 'is_thanksgiving', 'is_christmas', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'MarkDown1_exists', 'MarkDown2_exists', 'MarkDown3_exists', 'MarkDown4_exists', 'MarkDown5_exists', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
TFT static exogenous inputs: ['Size', 'Type_A', 'Type_B', 'Type_C']


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | valid_loss              | MAE                      | 0      | train
2  | hist_cat_embeddings     | ModuleList               | 0      | train
3  | futr_cat_embeddings     | ModuleList               | 0      | train
4  | stat_cat_embeddings     | ModuleList               | 0      | train
5  | padder_train            | ConstantPad1d            | 0      | train
6  | scaler                  | TemporalNorm             | 0      | train
7  | embedding               | TFTEmbedd

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

WMAE: 1923.8754
Holiday WMAE: 2195.1545
Non-holiday WMAE: 1819.3719
Fallback rows: 23 | unseen series: 10


,split,wmae,holiday_wmae,nonholiday_wmae,missing_prediction_rows_before_fallback,unseen_series_before_fallback
0,baseline_valid,2245.161477,2518.779489,2139.757027,NaN,NaN
1,valid,1923.875376,2195.154474,1819.371933,23.0,10.0


Improvement over baseline: 321.2861017959799
Improvement %: 14.310155639916731


## 10. MLflow / DagsHub logging

In [14]:
import dagshub
import mlflow
import mlflow.pyfunc

dagshub.init(
    repo_owner="izere23",
    repo_name="ML-Final-Walmart-Recruiting-Store-Sales-Forecasting",
    mlflow=True,
)

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("TFT_Training")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=deaae1bb-49c6-4288-b9a3-d76c24d83d0c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=c7374250409c0cec4a16b2d148b4f999a9c42dc88ba246a793ace14cd04a383d




Accessing as gvakh23

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

MLflow tracking URI: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow


## 11. Log validation run and NeuralForecast model artifact

In [15]:
import json
from pathlib import Path

EXPERIMENT_NAME = "TFT_Training"
ARTIFACT_DIR = Path("artifacts") / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

valid_preds.to_csv(ARTIFACT_DIR / "valid_predictions.csv", index=False)
valid_eval.to_csv(ARTIFACT_DIR / "valid_eval.csv", index=False)
summary.to_csv(ARTIFACT_DIR / "metrics_summary.csv", index=False)
if train_tail_eval is not None:
    train_tail_eval.to_csv(ARTIFACT_DIR / "train_tail_eval.csv", index=False)

feature_summary = {
    "experiment": EXPERIMENT,
    "target": CONFIG["target_col"],
    "future_exogenous": FUTR_EXOG_COLS,
    "static_exogenous": STAT_EXOG_COLS,
}

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=4)
with open(ARTIFACT_DIR / "feature_summary.json", "w") as f:
    json.dump(feature_summary, f, indent=4)

MODEL_DIR = ARTIFACT_DIR / "neuralforecast_model"
nf_valid.save(path=str(MODEL_DIR), overwrite=True)

if mlflow.active_run():
    mlflow.end_run()
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(CONFIG)
    mlflow.log_param("futr_exog_cols", ",".join(FUTR_EXOG_COLS))
    mlflow.log_param("stat_exog_cols", ",".join(STAT_EXOG_COLS))
    mlflow.log_param("n_futr_exog", len(FUTR_EXOG_COLS))
    mlflow.log_param("n_stat_exog", len(STAT_EXOG_COLS))
    mlflow.log_param("run_train_tail_check", RUN_TRAIN_TAIL_CHECK)

    mlflow.log_metric("baseline_store_dept_median_wmae", baseline_valid_metrics["wmae"])
    mlflow.log_metric("valid_wmae", valid_metrics["wmae"])
    mlflow.log_metric("holiday_wmae", valid_metrics["holiday_wmae"])
    mlflow.log_metric("nonholiday_wmae", valid_metrics["nonholiday_wmae"])
    mlflow.log_metric("missing_prediction_rows_before_fallback", valid_metrics["missing_prediction_rows_before_fallback"])
    mlflow.log_metric("unseen_series_before_fallback", valid_metrics["unseen_series_before_fallback"])
    mlflow.log_metric("improvement_over_baseline", improvement)
    mlflow.log_metric("improvement_over_baseline_pct", improvement_pct)

    if train_tail_metrics is not None:
        mlflow.log_metric("baseline_train_tail_wmae", baseline_train_tail_metrics["wmae"])
        mlflow.log_metric("train_tail_wmae", train_tail_metrics["wmae"])
        mlflow.log_metric("train_tail_holiday_wmae", train_tail_metrics["holiday_wmae"])
        mlflow.log_metric("train_tail_nonholiday_wmae", train_tail_metrics["nonholiday_wmae"])
        mlflow.log_metric("generalization_gap_valid_minus_train_tail", generalization_gap)

    mlflow.log_artifacts(str(ARTIFACT_DIR), artifact_path="artifacts")
    mlflow.log_artifacts(str(MODEL_DIR), artifact_path="neuralforecast_model")

print("✅ TFT validation run logged:", RUN_NAME)


🏃 View run TFT_all_exog_input104_h14_hidden64_heads4_lr0.001_drop0.05_scalerrobust_steps4000 at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/4/runs/ca40032340cd416685c9f05524410fbf
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/4
✅ TFT validation run logged: TFT_all_exog_input104_h14_hidden64_heads4_lr0.001_drop0.05_scalerrobust_steps4000


In [ ]:
# Confirm exactly what the trained model used.
valid_input_check, valid_static_check = make_tft_dfs(
    train_part,
    CONFIG,
    futr_exog_cols=FUTR_EXOG_COLS,
    stat_exog_cols=STAT_EXOG_COLS,
    include_y=True,
)
print("Validation temporal input columns:", valid_input_check.columns.tolist())
print("Validation static input columns:", [] if valid_static_check is None else valid_static_check.columns.tolist())


In [ ]:
display(valid_input_check.head())


## 12. Final model for Kaggle test + MLflow pyfunc wrapper

ეს ნაწილი მაშინ გაუშვი, როცა validation შედეგი მოგეწონება.

აქ ვაკეთებთ:

1. `CONFIG_FINAL["horizon"] = test_full["Date"].nunique()` — Kaggle test-ის ყველა კვირისთვის;
2. ვატრენინგებთ TFT-ს `train_full`-ზე;
3. ვაკეთებთ `mlflow.pyfunc.PythonModel` wrapper-ს;
4. wrapper-ის `predict(raw_test)` იღებს raw Kaggle `test.csv.zip`-ს, შიგნით თვითონ აერთებს `features/stores`, აკეთებს feature engineering-ს და აბრუნებს Kaggle submission-ს.

ეს უფრო ახლოსაა დავალების მოთხოვნასთან: **pipeline უნდა ეშვებოდეს პირდაპირ raw test set-ზე**.

In [ ]:
CONFIG_FINAL = CONFIG.copy()
CONFIG_FINAL["horizon"] = int(test_full["Date"].nunique())

FINAL_RUN_NAME = (
    f"TFT_FINAL_{CONFIG_FINAL['model_type']}"
    f"_input{CONFIG_FINAL['input_size']}_h{CONFIG_FINAL['horizon']}"
    f"_hidden{CONFIG_FINAL['hidden_size']}_heads{CONFIG_FINAL['n_head']}"
    f"_lr{CONFIG_FINAL['learning_rate']}_scaler{CONFIG_FINAL['scaler_type']}"
    f"_steps{CONFIG_FINAL['max_steps']}"
)
print(FINAL_RUN_NAME)
CONFIG_FINAL


In [ ]:
# Run only after selecting the final validation winner.
train_full_nf, train_full_static_df = make_tft_dfs(
    train_full,
    CONFIG_FINAL,
    futr_exog_cols=FUTR_EXOG_COLS,
    stat_exog_cols=STAT_EXOG_COLS,
    include_y=True,
)

nf_final = build_tft(CONFIG_FINAL, FUTR_EXOG_COLS, STAT_EXOG_COLS)
fit_kwargs = {"df": train_full_nf, "val_size": CONFIG_FINAL["horizon"]}
if train_full_static_df is not None:
    fit_kwargs["static_df"] = train_full_static_df
nf_final.fit(**fit_kwargs)

test_futr_df = make_complete_future_df(
    nf=nf_final,
    train_nf=train_full_nf,
    future_source_df=test_full,
    futr_exog_cols=FUTR_EXOG_COLS,
    horizon=CONFIG_FINAL["horizon"],
)
test_preds = nf_final.predict(futr_df=test_futr_df)
test_preds = inverse_target_transform(
    test_preds,
    model_col=CONFIG_FINAL["model"],
    target_transform=CONFIG_FINAL["target_transform"],
)

test_out = test_full.copy()
test_out["unique_id"] = test_out["Store"].astype(str) + "_" + test_out["Dept"].astype(str)
test_out["ds"] = pd.to_datetime(test_out["Date"])
test_out = test_out.merge(test_preds[["unique_id", "ds", "pred"]], on=["unique_id", "ds"], how="left")
test_out, test_fallback_rows, test_unseen_series = add_fallback_predictions(
    test_out,
    train_full,
    CONFIG_FINAL["target_col"],
)
test_out["Weekly_Sales"] = test_out["pred"].clip(lower=0)
test_out["Id"] = (
    test_out["Store"].astype(str) + "_" +
    test_out["Dept"].astype(str) + "_" +
    pd.to_datetime(test_out["Date"]).dt.strftime("%Y-%m-%d")
)

submission = test_out[["Id", "Weekly_Sales"]].copy()
submission.to_csv("submission_tft.csv", index=False)
display(submission.head())
print("Saved submission_tft.csv", submission.shape)
print("Test fallback rows:", test_fallback_rows, "| unseen series:", test_unseen_series)


In [ ]:
# Save artifacts needed by the pyfunc wrapper
FINAL_ARTIFACT_DIR = Path("artifacts") / FINAL_RUN_NAME
FINAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_NF_DIR = FINAL_ARTIFACT_DIR / "neuralforecast_model"
nf_final.save(path=str(FINAL_NF_DIR), overwrite=True)

with open(FINAL_ARTIFACT_DIR / "config_final.json", "w") as f:
    json.dump(CONFIG_FINAL, f, indent=4)

with open(FINAL_ARTIFACT_DIR / "feature_lists.json", "w") as f:
    json.dump(
        {"futr_exog_cols": FUTR_EXOG_COLS, "stat_exog_cols": STAT_EXOG_COLS},
        f,
        indent=4,
    )

# Copy raw side tables into artifacts so loaded model can preprocess raw test.csv by itself.
import shutil
shutil.copy(DATA_DIR / "features.csv.zip", FINAL_ARTIFACT_DIR / "features.csv.zip")
shutil.copy(DATA_DIR / "stores.csv", FINAL_ARTIFACT_DIR / "stores.csv")

submission.to_csv(FINAL_ARTIFACT_DIR / "submission_tft.csv", index=False)

print("Final artifacts:", FINAL_ARTIFACT_DIR)

In [ ]:
import mlflow
import mlflow.pyfunc
from neuralforecast import NeuralForecast

class WalmartTFTPyfunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import json
        import pandas as pd
        from neuralforecast import NeuralForecast

        self.nf = NeuralForecast.load(context.artifacts["nf_model"])

        with open(context.artifacts["config"], "r") as f:
            self.config = json.load(f)

        with open(context.artifacts["feature_lists"], "r") as f:
            lists = json.load(f)

        self.futr_exog_cols = lists["futr_exog_cols"]
        self.stat_exog_cols = lists["stat_exog_cols"]

        self.features = pd.read_csv(context.artifacts["features"])
        self.stores = pd.read_csv(context.artifacts["stores"])
        self.features["Date"] = pd.to_datetime(self.features["Date"])

    def _prepare_raw_test(self, raw_test):
        import numpy as np
        import pandas as pd

        test = raw_test.copy()
        test["Date"] = pd.to_datetime(test["Date"])

        # raw test has Store, Dept, Date, IsHoliday.
        # Merge the same raw side tables used in training.
        if "Temperature" not in test.columns:
            test = (
                test.merge(self.features, on=["Store", "Date", "IsHoliday"], how="left")
                    .merge(self.stores, on="Store", how="left")
            )

        # MarkDown flags
        markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
        for col in markdown_cols:
            test[col + "_exists"] = test[col].notna().astype(int)
            test[col] = test[col].fillna(0)

        # Calendar features
        test["Year"] = test["Date"].dt.year
        test["Month"] = test["Date"].dt.month
        test["WeekOfYear"] = test["Date"].dt.isocalendar().week.astype(int)
        test["Week_sin"] = np.sin(2 * np.pi * test["WeekOfYear"] / 52)
        test["Week_cos"] = np.cos(2 * np.pi * test["WeekOfYear"] / 52)

        super_bowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
        labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
        thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
        christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])

        test["is_superbowl"] = test["Date"].isin(super_bowl).astype(int)
        test["is_labor_day"] = test["Date"].isin(labor_day).astype(int)
        test["is_thanksgiving"] = test["Date"].isin(thanksgiving).astype(int)
        test["is_christmas"] = test["Date"].isin(christmas).astype(int)

        test["unique_id"] = test["Store"].astype(str) + "_" + test["Dept"].astype(str)
        test["ds"] = pd.to_datetime(test["Date"])
        test["IsHoliday_int"] = test["IsHoliday"].astype(bool).astype(int)

        for t in ["A", "B", "C"]:
            test[f"Type_{t}"] = (test["Type"].astype(str) == t).astype(int)

        for col in self.futr_exog_cols + self.stat_exog_cols:
            if col not in test.columns:
                test[col] = 0
            test[col] = pd.to_numeric(test[col], errors="coerce").fillna(0)

        futr_df = test[["unique_id", "ds"] + self.futr_exog_cols].sort_values(["unique_id", "ds"])
        return test, futr_df

    def predict(self, context, model_input):
        import pandas as pd

        test, futr_df = self._prepare_raw_test(model_input)
        if self.futr_exog_cols:
            preds = self.nf.predict(futr_df=futr_df)
        else:
            preds = self.nf.predict()
        preds = preds.rename(columns={self.config["model"]: "Weekly_Sales"})
        preds["Weekly_Sales"] = preds["Weekly_Sales"].clip(lower=0)

        out = test.merge(
            preds[["unique_id", "ds", "Weekly_Sales"]],
            on=["unique_id", "ds"],
            how="left",
        )
        out["Weekly_Sales"] = out["Weekly_Sales"].fillna(0).clip(lower=0)
        out["Id"] = (
            out["Store"].astype(str) + "_" +
            out["Dept"].astype(str) + "_" +
            pd.to_datetime(out["Date"]).dt.strftime("%Y-%m-%d")
        )
        return out[["Id", "Weekly_Sales"]].sort_values("Id").reset_index(drop=True)

In [ ]:
# Validate wrapper locally before logging
wrapper = WalmartTFTPyfunc()

# MLflow will provide artifacts through context, but for a local smoke test we use pyfunc after logging below.
raw_test_example = pd.read_csv(DATA_DIR / "test.csv.zip").head(200)
raw_test_example["Date"] = pd.to_datetime(raw_test_example["Date"])
raw_test_example.head()

In [ ]:
if mlflow.active_run():
    mlflow.end_run()

mlflow.set_experiment("TFT_Training")

artifacts = {
    "nf_model": str(FINAL_NF_DIR),
    "config": str(FINAL_ARTIFACT_DIR / "config_final.json"),
    "feature_lists": str(FINAL_ARTIFACT_DIR / "feature_lists.json"),
    "features": str(FINAL_ARTIFACT_DIR / "features.csv.zip"),
    "stores": str(FINAL_ARTIFACT_DIR / "stores.csv"),
}

input_example = raw_test_example

with mlflow.start_run(run_name=FINAL_RUN_NAME) as run:
    mlflow.log_params(CONFIG_FINAL)
    mlflow.log_param("futr_exog_cols", ",".join(FUTR_EXOG_COLS))
    mlflow.log_param("stat_exog_cols", ",".join(STAT_EXOG_COLS))

    mlflow.log_metric("validation_wmae_before_final_training", valid_metrics["wmae"])
    mlflow.log_metric("validation_holiday_wmae_before_final_training", valid_metrics["holiday_wmae"])
    mlflow.log_artifact(str(FINAL_ARTIFACT_DIR / "submission_tft.csv"), artifact_path="submissions")

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=WalmartTFTPyfunc(),
        artifacts=artifacts,
        input_example=input_example,
        pip_requirements=[
            "mlflow",
            "pandas",
            "numpy",
            "neuralforecast",
            "torch",
            "pytorch-lightning",
        ],
        registered_model_name="Walmart_TFT_Pipeline",
    )

print("✅ Final TFT pyfunc pipeline logged and registered.")
print("Model URI:", model_info.model_uri)

## 13. Inference smoke test from MLflow model

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
submission_from_loaded = loaded_model.predict(raw_test_example)

display(submission_from_loaded.head())
print(submission_from_loaded.shape)

## 14. model_inference.ipynb-ში გამოსაყენებელი მინიმალური კოდი

In [ ]:
# Minimal inference code for model_inference.ipynb

import pandas as pd
import mlflow.pyfunc

MODEL_URI = "models:/Walmart_TFT_Pipeline/latest"  # ან კონკრეტული version: models:/Walmart_TFT_Pipeline/1
DATA_DIR = "/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/data/raw"

raw_test = pd.read_csv(f"{DATA_DIR}/test.csv.zip")
model = mlflow.pyfunc.load_model(MODEL_URI)

submission = model.predict(raw_test)
submission.to_csv("submission_tft_from_registry.csv", index=False)

submission.head()

## 15. Experiment order

### Feature ablation — მხოლოდ `EXPERIMENT` შეცვალე

1. `calendar_only`
2. `calendar_static`
3. `calendar_static_markdown`
4. `all_exog`

ამ ოთხივე run-ზე:
- `RUN_TRAIN_TAIL_CHECK = False`
- სხვა CONFIG პარამეტრები უცვლელი დატოვე.
- გამარჯვებული აირჩიე `valid_wmae`-ით.

### Tuning — მხოლოდ feature-ablation გამარჯვებულზე

შემდეგ თითო run-ში მხოლოდ ერთი პარამეტრი შეცვალე:

1. `input_size`: 104 → 52
2. საჭიროების შემთხვევაში `input_size`: 52/104 → 78
3. `scaler_type`: robust → identity
4. `hidden_size`: 64 → 128 (`n_head=4` დატოვე)
5. საუკეთესო capacity-ზე `dropout`: 0.10 → 0.20
6. `learning_rate`: 0.001 → 0.0005

საბოლოო გამარჯვებულზე:
- `RUN_TRAIN_TAIL_CHECK = True`
- გაუშვი კიდევ ერთხელ seed 42-ით;
- შემდეგ stability check-ისთვის მხოლოდ `random_seed=123` შეცვალე.

Final Kaggle section მხოლოდ ამის შემდეგ გაუშვი.
